# PayShield AI — Model Training

## AI-Powered Payment Success Optimization

### Objective

Train a machine learning model to predict whether a receiver bank
is likely to enter a degraded or severe state within the next
15 minutes.

The model will use payment behavior and historical trends.

Primary evaluation metrics:

- Precision
- Recall
- F1-score
- PR-AUC
- ROC-AUC

Accuracy will not be the primary metric because the dataset is
highly imbalanced.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

In [2]:
df = pd.read_csv(
    "../data/processed/ml_dataset.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50944, 22)


,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_failure_rate,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,risk_target
0,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59,0,...,0.0,1447.700,0.0,0.0,-565.765,0.0,0.0,1164.817500,0.0,0
1,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00,1,...,0.0,881.935,0.0,0.0,0.945,0.0,0.0,1070.838333,0.0,0
2,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70,1,...,0.0,882.880,0.0,0.0,-131.210,0.0,0.0,838.828333,0.0,0
3,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14,1,...,0.0,751.670,0.0,0.0,410.250,0.0,0.0,932.156667,0.0,0
4,1,0.0,0.0,709.700,709.70,709.7000,0.0,920.47,920.47,2,...,0.0,1161.920,0.0,0.0,-452.220,0.0,0.0,874.430000,0.0,0


In [3]:
X = df.drop(columns=["risk_target"])
y = df["risk_target"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (50944, 21)
Target: (50944,)


In [4]:
print("Class distribution:")
print(y.value_counts())

print("\nPercentage:")
print(
    y.value_counts(normalize=True) * 100
)

Class distribution:
risk_target
0    50662
1      282
Name: count, dtype: int64

Percentage:
risk_target
0    99.446451
1     0.553549
Name: proportion, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training samples: 40755
Testing samples: 10189

Training target:
risk_target
0    40529
1      226
Name: count, dtype: int64

Testing target:
risk_target
0    10133
1       56
Name: count, dtype: int64


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
scaler.fit_transform(X_train)

array([[-0.81352751, -0.20094822, -0.17876839, ..., -0.33242645,
        -0.25543007, -0.30465751],
       [ 0.87265027, -0.20094822, -0.17876839, ..., -0.33242645,
        -0.35309332, -0.30465751],
       [-0.81352751, -0.20094822, -0.17876839, ..., -0.33242645,
         1.58187802, -0.30465751],
       ...,
       [ 1.71573915, -0.20094822, -0.17876839, ..., -0.33242645,
        -0.14042025, -0.30465751],
       [-0.81352751, -0.20094822, -0.17876839, ..., -0.33242645,
        -0.99616412, -0.30465751],
       [-0.81352751, -0.20094822, -0.17876839, ..., -0.33242645,
         4.09545185,  4.47697272]], shape=(40755, 21))

In [8]:
scaler.transform(X_test)

array([[ 1.71573915, -0.20094822, -0.17876839, ..., -0.33242645,
         0.68851144, -0.30465751],
       [ 0.02956138, -0.20094822, -0.17876839, ..., -0.33242645,
         0.59818489, -0.30465751],
       [ 0.02956138, -0.20094822, -0.17876839, ..., -0.33242645,
         0.0590717 , -0.30465751],
       ...,
       [ 0.87265027, -0.20094822, -0.17876839, ..., -0.33242645,
         0.51709702, -0.30465751],
       [ 0.02956138, -0.20094822, -0.17876839, ..., -0.33242645,
         0.27868163, -0.30465751],
       [ 0.02956138, -0.20094822, -0.17876839, ..., -0.33242645,
        -0.81219045, -0.30465751]], shape=(10189, 21))

In [9]:
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(
    X_train_scaled,
    y_train
)

print("Model trained successfully!")

Model trained successfully!


In [10]:
y_pred = model.predict(X_test_scaled)

y_probability = model.predict_proba(
    X_test_scaled
)[:, 1]

In [11]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No Risk",
            "Risk"
        ]
    )
)

              precision    recall  f1-score   support

     No Risk       1.00      0.98      0.99     10133
        Risk       0.19      0.88      0.31        56

    accuracy                           0.98     10189
   macro avg       0.59      0.93      0.65     10189
weighted avg       0.99      0.98      0.99     10189



In [12]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[9919  214]
 [   7   49]]


In [13]:
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.9492182543598708


In [14]:
pr_auc = average_precision_score(
    y_test,
    y_probability
)

print("PR-AUC:", pr_auc)

PR-AUC: 0.8333366504377651


In [15]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = (
    feature_importance
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

feature_importance

,feature,coefficient,absolute_coefficient
19,rolling_latency,1.418977,1.418977
11,is_weekend,0.920811,0.920811
10,day_of_week,-0.865368,0.865368
9,hour,-0.729550,0.729550
20,rolling_timeout_rate,-0.426885,0.426885
18,rolling_failure_rate,0.296738,0.296738
6,bank_error_rate,-0.245645,0.245645
2,timeout_rate,0.243214,0.243214
8,max_amount,0.237832,0.237832
7,avg_amount,-0.220316,0.220316


In [16]:
feature_importance.head(10)

,feature,coefficient,absolute_coefficient
19,rolling_latency,1.418977,1.418977
11,is_weekend,0.920811,0.920811
10,day_of_week,-0.865368,0.865368
9,hour,-0.729550,0.729550
20,rolling_timeout_rate,-0.426885,0.426885
18,rolling_failure_rate,0.296738,0.296738
6,bank_error_rate,-0.245645,0.245645
2,timeout_rate,0.243214,0.243214
8,max_amount,0.237832,0.237832
7,avg_amount,-0.220316,0.220316


In [17]:
results = X_test.copy()

results["actual_risk"] = y_test.values
results["risk_probability"] = y_probability

results = results.sort_values(
    "risk_probability",
    ascending=False
)

results.head(20)

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,actual_risk,risk_probability
46631,1,0.000000,0.000000,5652.450000,5652.45,5652.4500,0.000000,213.400000,213.40,13,...,8247.270000,50.000000,-100.000000,-2594.820000,-50.000000,33.333333,6147.548889,27.777778,1,1.000000
39835,1,0.000000,100.000000,6274.270000,6274.27,6274.2700,0.000000,1548.810000,1548.81,14,...,8254.210000,100.000000,-100.000000,-1979.940000,0.000000,66.666667,5475.016667,66.666667,1,1.000000
23673,2,0.000000,100.000000,6509.250000,7496.39,7397.6760,0.000000,219.560000,362.47,12,...,3866.790000,100.000000,0.000000,2642.460000,0.000000,22.222222,4974.198889,66.666667,1,1.000000
2414,2,50.000000,100.000000,6481.080000,6684.22,6663.9060,0.000000,240.870000,361.29,10,...,3367.414000,0.000000,10.000000,3113.666000,100.000000,30.000000,5207.868000,66.666667,1,1.000000
39868,1,0.000000,0.000000,5621.060000,5621.06,5621.0600,0.000000,1171.500000,1171.50,17,...,5154.710000,100.000000,-100.000000,466.350000,-100.000000,33.333333,4988.096667,33.333333,1,1.000000
16928,2,100.000000,0.000000,3213.280000,3778.72,3722.1760,100.000000,2500.510000,4923.39,18,...,7878.845000,100.000000,0.000000,-4665.565000,-100.000000,100.000000,4846.062778,44.444444,1,1.000000
2423,3,33.333333,33.333333,3802.976667,5478.34,5275.4180,0.000000,1174.206667,2983.36,11,...,4990.240000,66.666667,-33.333333,-1187.263333,-33.333333,33.333333,5151.255556,66.666667,1,1.000000
16921,2,50.000000,50.000000,3989.350000,4983.98,4884.5170,0.000000,268.125000,340.19,17,...,3506.536667,66.666667,-16.666667,482.813333,-16.666667,38.888889,5003.162222,72.222222,1,1.000000
16932,4,25.000000,0.000000,3263.782500,5553.29,5215.2065,25.000000,861.060000,1763.00,18,...,3347.457500,50.000000,0.000000,-83.675000,-50.000000,50.000000,4644.930000,16.666667,1,1.000000
16918,5,20.000000,20.000000,4106.212000,7066.63,6536.8280,20.000000,233.162000,378.48,17,...,4621.494000,40.000000,-80.000000,-515.282000,-20.000000,73.333333,4183.318667,20.000000,1,1.000000


In [18]:
def risk_level(probability):

    if probability >= 0.80:
        return "HIGH"

    elif probability >= 0.50:
        return "MEDIUM"

    else:
        return "LOW"


results["risk_level"] = (
    results["risk_probability"]
    .apply(risk_level)
)

results.head(20)

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,actual_risk,risk_probability,risk_level
46631,1,0.000000,0.000000,5652.450000,5652.45,5652.4500,0.000000,213.400000,213.40,13,...,50.000000,-100.000000,-2594.820000,-50.000000,33.333333,6147.548889,27.777778,1,1.000000,HIGH
39835,1,0.000000,100.000000,6274.270000,6274.27,6274.2700,0.000000,1548.810000,1548.81,14,...,100.000000,-100.000000,-1979.940000,0.000000,66.666667,5475.016667,66.666667,1,1.000000,HIGH
23673,2,0.000000,100.000000,6509.250000,7496.39,7397.6760,0.000000,219.560000,362.47,12,...,100.000000,0.000000,2642.460000,0.000000,22.222222,4974.198889,66.666667,1,1.000000,HIGH
2414,2,50.000000,100.000000,6481.080000,6684.22,6663.9060,0.000000,240.870000,361.29,10,...,0.000000,10.000000,3113.666000,100.000000,30.000000,5207.868000,66.666667,1,1.000000,HIGH
39868,1,0.000000,0.000000,5621.060000,5621.06,5621.0600,0.000000,1171.500000,1171.50,17,...,100.000000,-100.000000,466.350000,-100.000000,33.333333,4988.096667,33.333333,1,1.000000,HIGH
16928,2,100.000000,0.000000,3213.280000,3778.72,3722.1760,100.000000,2500.510000,4923.39,18,...,100.000000,0.000000,-4665.565000,-100.000000,100.000000,4846.062778,44.444444,1,1.000000,HIGH
2423,3,33.333333,33.333333,3802.976667,5478.34,5275.4180,0.000000,1174.206667,2983.36,11,...,66.666667,-33.333333,-1187.263333,-33.333333,33.333333,5151.255556,66.666667,1,1.000000,HIGH
16921,2,50.000000,50.000000,3989.350000,4983.98,4884.5170,0.000000,268.125000,340.19,17,...,66.666667,-16.666667,482.813333,-16.666667,38.888889,5003.162222,72.222222,1,1.000000,HIGH
16932,4,25.000000,0.000000,3263.782500,5553.29,5215.2065,25.000000,861.060000,1763.00,18,...,50.000000,0.000000,-83.675000,-50.000000,50.000000,4644.930000,16.666667,1,1.000000,HIGH
16918,5,20.000000,20.000000,4106.212000,7066.63,6536.8280,20.000000,233.162000,378.48,17,...,40.000000,-80.000000,-515.282000,-20.000000,73.333333,4183.318667,20.000000,1,1.000000,HIGH


In [19]:
results.to_csv(
    "../data/processed/model_predictions.csv",
    index=False
)

print("Predictions saved successfully!")

Predictions saved successfully!
